In [1]:
#!pip install langchain==0.3.10  #Installs LangChain framework which helps build the complete RAG pipeline

In [2]:
#!pip install langchain_openai==0.2.12

In [3]:
#!pip install langchain_community==0.3.11  #Provides community integrations like document loaders and external connectors.
#!pip install -U langchain-openai

In [4]:
#!pip install redis==5.2.0

In [6]:
#!pip install "Unstructured[pdf]==0.16.25"

In [9]:
#install ocr dependencies for unstructured
#!sudo apt-get install tesseract-ocr
#!sudo apt-get install poppler-utils

In [10]:
#!pip install htmltabletomd==1.0.0

In [12]:
#!pip install pypdf                        #Enables reading and extracting content from PDF files.
#!pip install langchain-text-splitters     #Used for splitting large documents into smaller chunks for efficient retrieval

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [1]:
#!pip install -U langchain-chroma chromadb

In [3]:
#!pip install -U langchain-openai   #Provides OpenAI integration for embeddings and LLM generation

In [4]:
#to prevent 403 errors with unstructured.io till they fix it
import nltk
nltk.download("pukit")
nltk.download("pukit_tab")
nltk.download("averaged_perceptron_tagger")

[nltk_data] Error loading pukit: Package 'pukit' not found in index
[nltk_data] Error loading pukit_tab: Package 'pukit_tab' not found in
[nltk_data]     index
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [6]:
#!pip install openai

In [7]:
# Entering API Key

from getpass import getpass

openai_key = getpass("Enter OPENAI API KEY: ")

Enter OPENAI API KEY: ··········


In [8]:
# Set Environment Variable ONCE

import os
import openai
os.environ["OPENAI_API_KEY"] = openai_key

In [9]:
from IPython.display import display,Markdown

In [13]:
#DOCUMENT PREPROCESSING (pdf's loading)
from google.colab import files

uploaded = files.upload()

Saving 1706.03762v7.pdf to 1706.03762v7.pdf
Saving 2005.11401v4.pdf to 2005.11401v4.pdf
Saving 2005.14165v4.pdf to 2005.14165v4.pdf


In [11]:
from langchain_community.document_loaders import PyPDFLoader

In [14]:
loader = PyPDFLoader(file_path = "/content/1706.03762v7.pdf")
loader = PyPDFLoader(file_path = "/content/2005.11401v4.pdf")
loader = PyPDFLoader(file_path = "/content/2005.14165v4.pdf")

In [15]:
def create_simple_chunks(file_path,
                         chunk_size=3000,
                         chunk_overlap=300):

    print("Loading pages:", file_path)

    loader = PyPDFLoader(file_path)
    doc_pages = loader.load()

    print("Chunking pages...")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    doc_chunks = splitter.split_documents(doc_pages)

    return doc_chunks

In [16]:
from glob import glob

pdf_files = glob("/content/*.pdf")

print(pdf_files)
paper_docs = []    #creates empty storage where loaded documents/pages will later be added

['/content/2005.11401v4.pdf', '/content/2005.14165v4.pdf', '/content/1706.03762v7.pdf']


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [19]:
for fp in pdf_files:
  paper_docs.extend(create_simple_chunks(file_path=fp,
                                         chunk_size=300, chunk_overlap=300))

Loading pages: /content/2005.11401v4.pdf
Chunking pages...
Loading pages: /content/2005.14165v4.pdf
Chunking pages...
Loading pages: /content/1706.03762v7.pdf
Chunking pages...


In [20]:
#combining all documents in one list for embedding and storing into vector database
total_docs = paper_docs   #already contains chunks from all 3 PDFs

print(len(total_docs))

3414


In [21]:
print(type(total_docs))
print(len(total_docs))

<class 'list'>
3414


In [22]:
from langchain_openai import OpenAIEmbeddings

openai_embed_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embedding model created")

Embedding model created


In [23]:
print(len(total_docs))

3414


In [25]:
from langchain_chroma import Chroma

chroma_db = Chroma.from_documents(
    documents=total_docs,
    embedding=openai_embed_model,
    collection_name="my_db",
    collection_metadata={"hnsw:space":"cosine"},
    persist_directory="/content/my_chromadb"
)

print("Chroma DB created successfully")

Chroma DB created successfully


In [26]:
#load vector db form disk
chroma_db = Chroma(
    persist_directory="/content/my_chromadb",
    embedding_function=openai_embed_model,
    collection_name="my_db"
)

chroma_db

In [27]:
#RETRIEVAL SYSTEM
# Semantic Similarity Based Retrieval

Similarity_retriever = chroma_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

print("Retriever created successfully")

Retriever created successfully


In [28]:
from IPython.display import display, Markdown

def display_docs(docs):
  for doc in docs:
    print("Metadata:",doc.metadata)
    display(Markdown(doc.page_content[:1000]))
    print()

In [29]:
query = "What are the main components of a RAG model, and how do they interact?"

top_docs = Similarity_retriever.invoke(query)

display_docs(top_docs)

Metadata: {'page': 1, 'source': '/content/2005.11401v4.pdf'}


1Code to run experiments with RAG has been open-sourced as part of the HuggingFace Transform-
ers Library [66] and can be found at https://github.com/huggingface/transformers/blob/master/
examples/rag/. An interactive demo of RAG models can be found at https://huggingface.co/rag/
2


Metadata: {'source': '/content/2005.11401v4.pdf', 'page': 16}


run experiments with RAG can be found athttps://github.com/huggingface/transformers/
blob/master/examples/rag/README.md and an interactive demo of a RAG model can be found
at https://huggingface.co/rag/
2https://github.com/pytorch/fairseq
3https://github.com/huggingface/transformers
17


Metadata: {'source': '/content/2005.11401v4.pdf', 'page': 0}


(RAG) — models which combine pre-trained parametric and non-parametric mem-
ory for language generation. We introduce RAG models where the parametric
memory is a pre-trained seq2seq model and the non-parametric memory is a dense


Metadata: {'page': 4, 'source': '/content/2005.11401v4.pdf'}


tasks, RAG sets a new state of the art (only on the T5-comparable split for TQA). RAG combines
the generation ﬂexibility of the “closed-book” (parametric only) approaches and the performance of
"open-book" retrieval-based approaches. Unlike REALM and T5+SSM, RAG enjoys strong results


Metadata: {'page': 2, 'source': '/content/2005.11401v4.pdf'}


2.1 Models
RAG-Sequence Model The RAG-Sequence model uses the same retrieved document to generate
the complete sequence. Technically, it treats the retrieved document as a single latent variable that
is marginalized to get the seq2seq probability p(y|x) via a top-K approximation. Concretely, the

In [30]:
#ANSWER GENERATION
# Building augmentation part (enhanced prompt)

from langchain_core.prompts import ChatPromptTemplate

rag_prompt = """
you are an assistant specialized in question answerinng and translation.
your task is to respond to the given question using only the information provided in the retrieved context.

Instructions:
- If the answer is not present in the context, clearly state:"I don't know based on the given Context."
- Do not invent or assume any information
- write the answer in clear, simple language with correct grammer.
- Make the response detailed, structured, and easy to understand.

Question:
{question}

context:
{context}

Answer:
"""

rag_prompt_template = ChatPromptTemplate.from_template(
    rag_prompt
)

In [31]:
# Last part: LLM and Generation
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": Similarity_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt_template
    | ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0
    )
)

In [32]:
#SOURCE ATTRIBUTION
query = "What are the main components of a RAG model, and how do they interact?"

result = rag_chain.invoke(query)

print(result.content)

The main components of a RAG (Retrieval-Augmented Generation) model are:

1. **Parametric Memory**: This is a pre-trained sequence-to-sequence (seq2seq) model. It is responsible for generating language based on the information it has learned during training.

2. **Non-Parametric Memory**: This component consists of a dense retrieval system that allows the model to access external information. It helps the model retrieve relevant documents or data that can enhance the generation process.

The interaction between these components occurs as follows:

- The RAG model combines the strengths of both parametric and non-parametric memories. The parametric memory provides the generation flexibility typical of "closed-book" approaches, where the model relies solely on its training data. In contrast, the non-parametric memory allows the model to perform "open-book" retrieval, accessing additional information to improve its responses.

- Specifically, in the RAG-Sequence model, the same retrieved 

In [33]:
query = "What are the two sub-layers in each encoder layer of the Transformer model?"

result = rag_chain.invoke(query)

print(result.content)

The two sub-layers in each encoder layer of the Transformer model are:

1. A multi-head self-attention mechanism.
2. A position-wise fully connected feed-forward network. 

These sub-layers are applied to each position separately and identically, and residual connections are used around each of them.


In [34]:
query = "Explain how positional encoding is implemented in Transformers and why it is necessary."

result = rag_chain.invoke(query)

print(result.content)

Positional encoding in Transformers is implemented by adding positional encodings to the input embeddings at the bottom of both the encoder and decoder stacks. This is necessary because Transformers do not have recurrence or convolution, which means they do not inherently understand the order of the sequence of tokens. 

To address this, positional encodings provide information about the relative or absolute positions of the tokens within the sequence. The positional encodings are designed to have the same dimension, referred to as d_model, as the embeddings. This allows the two to be summed together, enabling the model to incorporate positional information into its processing.

There are various methods for creating positional encodings, including both learned and fixed encodings. In the context provided, sine and cosine functions of different frequencies are used for the positional encodings. This approach helps the model to effectively utilize the order of the sequence in its comput

In [35]:
query = "Describe the concept of multi-head attention in the Transformer architecture. Why is it beneficial?"

result = rag_chain.invoke(query)

print(result.content)

Multi-head attention is a key component of the Transformer architecture that allows the model to focus on different parts of the input sequence simultaneously. In this approach, multiple attention heads are used, each with its own set of learned parameters. This means that each head can capture different relationships and dependencies in the data.

The Transformer employs multi-head attention in three main ways, one of which is in the "encoder-decoder attention" layers. In this case, the queries come from the previous decoder layer, while the memory keys and values are derived from the encoder's output. This setup enables the model to effectively integrate information from both the encoder and decoder.

The benefits of multi-head attention include:

1. **Parallel Processing**: By using multiple heads, the model can process different parts of the input at the same time, which enhances its ability to learn complex patterns and relationships.

2. **Improved Learning of Dependencies**: Mul

In [36]:
query = "What is few-shot learning, and how does GPT-3 implement it during inference?"

result = rag_chain.invoke(query)

print(result.content)

Few-shot learning is a setting in which a model, like GPT-3, is provided with a small number of examples or demonstrations to help it understand and perform a specific task. In the case of GPT-3, this is done without any gradient updates or fine-tuning. Instead, tasks and few-shot demonstrations are specified purely through text interaction with the model.

During inference, GPT-3 implements few-shot learning by allowing as many demonstrations as will fit into its context window, which typically accommodates between 10 to 100 examples. This process is referred to as in-context learning. However, there is some uncertainty regarding whether GPT-3 learns new tasks from scratch during this process or if it simply recognizes and identifies tasks based on the provided examples.

In summary, few-shot learning in GPT-3 involves using a limited number of text-based demonstrations to guide the model's performance on various tasks, without modifying the model itself.
